# 04 — Zero-Shot v1 Base

**Amostra:** `data/sample/toldBr_sample_500.csv` (500 tweets estratificados)

Prompt mínimo — apenas nomes das categorias, sem descrições, sem anti-bias.

- Saída: `results/zero_shot_v1_base.csv`

In [1]:
import time
from pathlib import Path
import polars as pl
import requests
from sklearn.metrics import classification_report, f1_score

OLLAMA_BASE = "http://127.0.0.1:11434"
MODEL = "qwen3.5:9b"
VALID_LABELS = {"NOT_TOXIC", "OBSCENE", "INSULT", "HOMOPHOBIA", "RACISM", "MISOGYNY", "XENOPHOBIA"}
OUTPUT = "../results/zero_shot_v1_base.csv"

Path("../results").mkdir(exist_ok=True)
sample = pl.read_csv("../data/sample/toldBr_sample_500.csv")
print(f"Tweets: {len(sample)}")

Tweets: 500


In [2]:
def build_prompt(tweet: str) -> str:
    return (
        "Você é um sistema de moderação de conteúdo em português brasileiro.\n"
        "Classifique o comentário abaixo em UMA das categorias:\n"
        "NOT_TOXIC, OBSCENE, INSULT, HOMOPHOBIA, RACISM, MISOGYNY, XENOPHOBIA\n"
        "Responda APENAS com o nome da categoria.\n\n"
        f"Comentário: {tweet}\n"
        "Classificação:"
    )

def parse_label(response: str) -> str:
    text = response.strip().upper().replace(" ", "_")
    for label in VALID_LABELS:
        if label in text:
            return label.lower()
    return "unknown"

In [3]:
resultados = []
t0 = time.time()

for i, row in enumerate(sample.iter_rows(named=True)):
    payload = {"model": MODEL, "prompt": build_prompt(row["text"]), "stream": False, "think": False}
    r = requests.post(f"{OLLAMA_BASE}/api/generate", json=payload)
    data = r.json()
    predicao = parse_label(data["response"])
    tps = data["eval_count"] / (data["eval_duration"] / 1e9)
    resultados.append({"text": row["text"], "label": row["label"], "predicao": predicao,
                       "resposta_raw": data["response"].strip(), "tokens_s": round(tps, 1)})
    if (i + 1) % 50 == 0:
        print(f"{i+1}/500 — {time.time()-t0:.0f}s")

df = pl.DataFrame(resultados)
df.write_csv(OUTPUT)
print(f"\nConcluído em {time.time()-t0:.0f}s | UNKNOWN: {(df['predicao']=='unknown').sum()}")

50/500 — 17s


100/500 — 29s


150/500 — 40s


200/500 — 52s


250/500 — 64s


300/500 — 76s


350/500 — 88s


400/500 — 99s


450/500 — 111s


500/500 — 123s

Concluído em 123s | UNKNOWN: 0


In [4]:
y_true, y_pred = df["label"].to_list(), df["predicao"].to_list()
f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
print(f"F1-macro: {f1:.4f}\n")
print(classification_report(y_true, y_pred, zero_division=0))

F1-macro: 0.2347

              precision    recall  f1-score   support

  homophobia       0.40      0.50      0.44         4
      insult       0.20      0.06      0.09        36
    misogyny       0.00      0.00      0.00         1
   not_toxic       0.83      0.90      0.86       403
     obscene       0.33      0.20      0.25        55
      racism       0.00      0.00      0.00         0
  xenophobia       0.00      0.00      0.00         1

    accuracy                           0.75       500
   macro avg       0.25      0.24      0.23       500
weighted avg       0.72      0.75      0.73       500

